In [0]:
%sql

CREATE TABLE IF NOT EXISTS workspace.retail_training.pipeline_audit_log (
    pipeline_name STRING,
    task_name STRING,
    run_status STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    rows_processed BIGINT,
    error_message STRING,
    logged_at TIMESTAMP
)
USING DELTA;


In [0]:

import uuid
from datetime import datetime, timezone

from pyspark.sql.types import (
    StructType, StructField, StringType,
    TimestampType, LongType
)

pipeline_name = "retail_data_pipeline"
task_name = "08_pipeline_monitoring"
audit_table = "workspace.retail_training.pipeline_audit_log"

run_id = str(uuid.uuid4())
start_time = datetime.now(timezone.utc)

print(f"Pipeline run ID: {run_id}")
print("Monitoring audit initialized.")


def write_audit_record(
    run_status,
    rows_processed=None,
    error_message=None
):
    end_time = datetime.now(timezone.utc)

    audit_schema = StructType([
        StructField("pipeline_name", StringType(), False),
        StructField("task_name", StringType(), False),
        StructField("run_id", StringType(), True),
        StructField("run_status", StringType(), False),
        StructField("start_time", TimestampType(), True),
        StructField("end_time", TimestampType(), True),
        StructField("rows_processed", LongType(), True),
        StructField("error_message", StringType(), True),
        StructField("logged_at", TimestampType(), True)
    ])

    audit_record = [(
        pipeline_name,
        task_name,
        run_id,
        run_status,
        start_time,
        end_time,
        rows_processed,
        error_message[:4000] if error_message else None,
        end_time
    )]

    audit_df = spark.createDataFrame(
        audit_record,
        schema=audit_schema
    )

    audit_df.write.format("delta").mode("append").saveAsTable(
        audit_table
    )


Pipeline run ID: b31e5ee8-3a5c-4686-b58c-ad7ce94a32f5
Monitoring audit initialized.


In [0]:

from datetime import datetime, timezone

pipeline_name = "retail_data_pipeline"

tables_to_monitor = {
    "bronze_customers": "workspace.retail_training.bronze_customers",
    "bronze_products": "workspace.retail_training.bronze_products",
    "bronze_stores": "workspace.retail_training.bronze_stores",
    "bronze_sales": "workspace.retail_training.bronze_sales",
    "silver_sales": "workspace.retail_training.silver_sales",
    "gold_monthly_sales": "workspace.retail_training.gold_monthly_sales",
    "gold_product_sales": "workspace.retail_training.gold_product_sales",
    "gold_store_sales": "workspace.retail_training.gold_store_sales"
}

monitoring_start = datetime.now(timezone.utc)
table_metrics = []

# Count each table only once
for table_name, table_path in tables_to_monitor.items():
    row_count = spark.table(table_path).count()

    table_metrics.append({
        "table_name": table_name,
        "row_count": row_count
    })

monitoring_end = datetime.now(timezone.utc)

duration_seconds = (
    monitoring_end - monitoring_start
).total_seconds()

print(f"Tables monitored: {len(table_metrics)}")
print(f"Monitoring duration: {duration_seconds:.2f} seconds")

# Display the collected metrics
display(spark.createDataFrame(table_metrics))


Tables monitored: 8
Monitoring duration: 4.74 seconds


row_count,table_name
20,bronze_customers
12,bronze_products
5,bronze_stores
110,bronze_sales
110,silver_sales
4,gold_monthly_sales
12,gold_product_sales
5,gold_store_sales


In [0]:
%sql

CREATE TABLE IF NOT EXISTS workspace.retail_training.pipeline_table_metrics (
    pipeline_name STRING,
    table_name STRING,
    row_count BIGINT,
    monitoring_start TIMESTAMP,
    monitoring_end TIMESTAMP,
    duration_seconds DOUBLE,
    logged_at TIMESTAMP
)
USING DELTA;


In [0]:

from pyspark.sql.types import (
    StructType, StructField, StringType,
    LongType, TimestampType, DoubleType
)

metrics_table = "workspace.retail_training.pipeline_table_metrics"

# Reuse the metrics collected in Cell 3
metrics_schema = StructType([
    StructField("pipeline_name", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("row_count", LongType(), False),
    StructField("monitoring_start", TimestampType(), False),
    StructField("monitoring_end", TimestampType(), True),
    StructField("duration_seconds", DoubleType(), True),
    StructField("logged_at", TimestampType(), True)
])

completed_metrics = [
    (
        pipeline_name,
        metric["table_name"],
        metric["row_count"],
        monitoring_start,
        monitoring_end,
        duration_seconds,
        monitoring_end
    )
    for metric in table_metrics
]

metrics_df = spark.createDataFrame(
    completed_metrics,
    schema=metrics_schema
)

# Save the metrics to Delta
metrics_df.write.format("delta").mode("append").saveAsTable(
    metrics_table
)

print(f"SUCCESS: Saved {len(completed_metrics)} table metrics.")
print(f"Monitoring duration: {duration_seconds:.2f} seconds")

display(metrics_df)


SUCCESS: Saved 8 table metrics.
Monitoring duration: 4.74 seconds


pipeline_name,table_name,row_count,monitoring_start,monitoring_end,duration_seconds,logged_at
retail_data_pipeline,bronze_customers,20,2026-10-21T03:17:36.380Z,2026-10-21T03:17:41.118Z,4.738423,2026-10-21T03:17:41.118Z
retail_data_pipeline,bronze_products,12,2026-10-21T03:17:36.380Z,2026-10-21T03:17:41.118Z,4.738423,2026-10-21T03:17:41.118Z
retail_data_pipeline,bronze_stores,5,2026-10-21T03:17:36.380Z,2026-10-21T03:17:41.118Z,4.738423,2026-10-21T03:17:41.118Z
retail_data_pipeline,bronze_sales,110,2026-10-21T03:17:36.380Z,2026-10-21T03:17:41.118Z,4.738423,2026-10-21T03:17:41.118Z
retail_data_pipeline,silver_sales,110,2026-10-21T03:17:36.380Z,2026-10-21T03:17:41.118Z,4.738423,2026-10-21T03:17:41.118Z
retail_data_pipeline,gold_monthly_sales,4,2026-10-21T03:17:36.380Z,2026-10-21T03:17:41.118Z,4.738423,2026-10-21T03:17:41.118Z
retail_data_pipeline,gold_product_sales,12,2026-10-21T03:17:36.380Z,2026-10-21T03:17:41.118Z,4.738423,2026-10-21T03:17:41.118Z
retail_data_pipeline,gold_store_sales,5,2026-10-21T03:17:36.380Z,2026-10-21T03:17:41.118Z,4.738423,2026-10-21T03:17:41.118Z


In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.retail_training.pipeline_quality_results (
    pipeline_name STRING,
    check_name STRING,
    table_name STRING,
    expected STRING,
    actual_value STRING,
    status STRING,
    checked_at TIMESTAMP
)
USING DELTA;

In [0]:

from datetime import datetime, timezone
from pyspark.sql import functions as F

pipeline_name = "retail_data_pipeline"
schema_name = "workspace.retail_training"

# Define the data quality checks
checks = [
    {
        "check_name": "bronze_sales_not_empty",
        "table_name": "bronze_sales",
        "check_type": "row_count",
        "expected": "greater_than_zero"
    },
    {
        "check_name": "silver_sales_not_empty",
        "table_name": "silver_sales",
        "check_type": "row_count",
        "expected": "greater_than_zero"
    },
    {
        "check_name": "silver_sales_unique_sale_id",
        "table_name": "silver_sales",
        "check_type": "unique_sale_id",
        "expected": "zero_null_and_duplicate_ids"
    },
    {
        "check_name": "gold_monthly_sales_not_empty",
        "table_name": "gold_monthly_sales",
        "check_type": "row_count",
        "expected": "greater_than_zero"
    },
    {
        "check_name": "gold_product_sales_not_empty",
        "table_name": "gold_product_sales",
        "check_type": "row_count",
        "expected": "greater_than_zero"
    },
    {
        "check_name": "gold_store_sales_not_empty",
        "table_name": "gold_store_sales",
        "check_type": "row_count",
        "expected": "greater_than_zero"
    }
]

check_results = []
check_time = datetime.now(timezone.utc)

for check in checks:
    table_path = f"{schema_name}.{check['table_name']}"

    if check["check_type"] == "row_count":
        row_count = spark.table(table_path).count()

        passed = row_count > 0
        actual_value = str(row_count)

    elif check["check_type"] == "unique_sale_id":
        sales_df = spark.table(table_path)

        # Count records with null sale IDs
        null_ids = sales_df.filter(
            F.col("sale_id").isNull()
        ).count()

        # Count distinct sale IDs that occur more than once
        duplicate_ids = (
            sales_df
            .filter(F.col("sale_id").isNotNull())
            .groupBy("sale_id")
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        passed = null_ids == 0 and duplicate_ids == 0

        actual_value = (
            f"null_ids={null_ids}, "
            f"duplicate_ids={duplicate_ids}"
        )

    else:
        raise ValueError(
            f"Unsupported check type: {check['check_type']}"
        )

    check_results.append({
        "pipeline_name": pipeline_name,
        "check_name": check["check_name"],
        "table_name": check["table_name"],
        "expected": check["expected"],
        "actual_value": actual_value,
        "status": "PASS" if passed else "FAIL",
        "checked_at": check_time
    })

# Display results for review
results_df = spark.createDataFrame(check_results)
display(results_df)

# Summarize check outcomes
failed_count = sum(
    1 for result in check_results
    if result["status"] == "FAIL"
)

print(f"Total checks: {len(check_results)}")
print(f"Passed: {len(check_results) - failed_count}")
print(f"Failed: {failed_count}")


actual_value,check_name,checked_at,expected,pipeline_name,status,table_name
110,bronze_sales_not_empty,2026-10-21T03:17:51.173Z,greater_than_zero,retail_data_pipeline,PASS,bronze_sales
110,silver_sales_not_empty,2026-10-21T03:17:51.173Z,greater_than_zero,retail_data_pipeline,PASS,silver_sales
"null_ids=0, duplicate_ids=0",silver_sales_unique_sale_id,2026-10-21T03:17:51.173Z,zero_null_and_duplicate_ids,retail_data_pipeline,PASS,silver_sales
4,gold_monthly_sales_not_empty,2026-10-21T03:17:51.173Z,greater_than_zero,retail_data_pipeline,PASS,gold_monthly_sales
12,gold_product_sales_not_empty,2026-10-21T03:17:51.173Z,greater_than_zero,retail_data_pipeline,PASS,gold_product_sales
5,gold_store_sales_not_empty,2026-10-21T03:17:51.173Z,greater_than_zero,retail_data_pipeline,PASS,gold_store_sales


Total checks: 6
Passed: 6
Failed: 0


In [0]:

from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType
)

quality_table = "workspace.retail_training.pipeline_quality_results"

quality_schema = StructType([
    StructField("pipeline_name", StringType(), False),
    StructField("check_name", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("expected", StringType(), False),
    StructField("actual_value", StringType(), False),
    StructField("status", StringType(), False),
    StructField("checked_at", TimestampType(), False)
])

quality_df = spark.createDataFrame(
    [
        (
            result["pipeline_name"],
            result["check_name"],
            result["table_name"],
            result["expected"],
            result["actual_value"],
            result["status"],
            result["checked_at"]
        )
        for result in check_results
    ],
    schema=quality_schema
)

quality_df.write.format("delta").mode("append").saveAsTable(
    quality_table
)

print(f"SUCCESS: Saved {quality_df.count()} quality check results.")

display(quality_df)


SUCCESS: Saved 6 quality check results.


pipeline_name,check_name,table_name,expected,actual_value,status,checked_at
retail_data_pipeline,bronze_sales_not_empty,bronze_sales,greater_than_zero,110,PASS,2026-10-21T03:17:51.173Z
retail_data_pipeline,silver_sales_not_empty,silver_sales,greater_than_zero,110,PASS,2026-10-21T03:17:51.173Z
retail_data_pipeline,silver_sales_unique_sale_id,silver_sales,zero_null_and_duplicate_ids,"null_ids=0, duplicate_ids=0",PASS,2026-10-21T03:17:51.173Z
retail_data_pipeline,gold_monthly_sales_not_empty,gold_monthly_sales,greater_than_zero,4,PASS,2026-10-21T03:17:51.173Z
retail_data_pipeline,gold_product_sales_not_empty,gold_product_sales,greater_than_zero,12,PASS,2026-10-21T03:17:51.173Z
retail_data_pipeline,gold_store_sales_not_empty,gold_store_sales,greater_than_zero,5,PASS,2026-10-21T03:17:51.173Z


In [0]:

failed_count = sum(
    1 for result in check_results
    if result["status"] == "FAIL"
)

try:
    if failed_count > 0:
        raise RuntimeError(
            f"DATA QUALITY FAILURE: {failed_count} check(s) failed. "
            "Review pipeline_quality_results."
        )

    # Record success only after all six checks pass.
    write_audit_record(
        run_status="SUCCESS",
        rows_processed=len(check_results),
        error_message=None
    )

    print("SUCCESS: All data quality checks passed.")
    print(
        f"SUCCESS audit record written: "
        f"{len(check_results)} quality checks completed."
    )

except Exception as e:
    try:
        write_audit_record(
            run_status="FAILED",
            rows_processed=len(check_results),
            error_message=str(e)
        )
        print("FAILED audit record written.")
    except Exception as audit_error:
        print(
            f"WARNING: Could not write failure audit record: "
            f"{audit_error}"
        )

    raise


SUCCESS: All data quality checks passed.
SUCCESS audit record written: 6 quality checks completed.
